In [23]:
from litellm import completion
from dotenv import load_dotenv
import json, os
from groq import Groq
#from pricer.batch import Batch
from pricer.items import Item
import os
load_dotenv(override=True)
MODEL = "groq/openai/gpt-oss-20b"
api_key = os.getenv('GROQ_API_KEY')
# we will pull data from Hugging face dataset that was pushed in DataCurate.ipynb 
# Use LITE_MODE = True for the free, fast version with training data size of 20,000
# USe LITE_MODE =  False for the powerful, full version with training data size of 800,000
LITE_MODE = True


In [24]:
#username = "ed-donner"
username = "allanhadoop"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"
train, val, test = Item.from_hub(dataset)

items = train + val + test
print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [25]:
for index, item in enumerate(items):
    item.id = index
items[2].id
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [4]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [26]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", api_key=api_key, reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Title: Schlage Interior Deadbolt Knob – Oil Rubbed Bronze  
Category: Hardware & Locks  
Brand: Schlage  
Description: A precision‑engineered interior knob that includes a deadbolt for added security.  
Details: Features an oil‑rubbed bronze finish, easy installation, and a lifetime mechanical and finish warranty.

Input tokens: 446
Output tokens: 98
Cost: 0.006 cents


In [27]:
#jsonl - This will make json a single line output 
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)
    
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "groq/openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two pi

In [28]:
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [29]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [ ]:
##### Batch processing using Groq -------This is paid service - https://console.groq.com/docs/batch#model-availability-and-pricing-----
## To run for entire 20k (Item lite) and 800k (Item full), there is batch.py program that runs below logic and generate outfile file which is loaded
## at Hugging face dataset 
groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(
        file=f,
        purpose="batch"
    )

response
file_id = response.id
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
result = groq.batches.retrieve(response.id)
result
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")
with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary
print(items[1000].summary)

In [ ]:
## Batch.py logic is as below - 
# Divides items into groups of 1,000
# Kicks off batches for each
# Allows us to monitor and collect the results when complete

Batch.create(items, LITE_MODE)
Batch.run()
Batch.fetch()
print(items[10234].summary)
# Remove the fields that we don't need in the hub
for item in items:
    item.full = None
    item.id = None

# Push the final dataset to the hub
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)


# Processed Data- 
# https://huggingface.co/datasets/ed-donner/items_lite
# https://huggingface.co/datasets/ed-donner/items_full